## tl;dr

- A v6 materializou **118 correções reais**: todas tiveram delta pareado positivo, com ganho médio de `+0,023949` no score do mesmo contrato.
- O salto posterior do score, de `0,771022041` (run 382) para `0,771110316` (run 383), **não mede nova qualidade textual**: os 287.895 textos são idênticos, mas os mesmos 118 itens mudaram de score porque a materialização alterou o baseline `OLD`, que faz parte das features do modelo `structural_v3`.
- O diagnóstico terminou, mas deixou a epoch 13 em `scored`; como baseline e output são idênticos, não há candidatos pareados, calibração nova nem promoções.
- Próxima evolução recomendada: separar score intrínseco de qualidade, risco de promoção e estado de release; depois reabrir o backlog de dívida de qualidade independentemente de o pacote estar fechado.

## Context & Methods

Análise pós-materialização para decidir a próxima etapa do sistema. Fonte principal: SQLite `memory/translation_engine.sqlite`, consultado em modo somente leitura. A unidade de análise é o segmento ativo; comparações de score usam os runs 381 (baseline v5), 382 (output antes da materialização) e 383 (mesmo texto após materializar a v6).

### Key Assumptions

- `package_version_changes` representa o delta textual congelado entre versões.
- Scores só são interpretados como comparáveis quando modelo, regra e contexto de features são equivalentes.
- O score do classificador é um indicador de risco operacional, não uma medida causal de qualidade linguística.

## Data

In [1]:
import json
import sqlite3
from pathlib import Path

workspace = Path.cwd()
if not (workspace / 'memory' / 'translation_engine.sqlite').exists():
    workspace = workspace.parent
db_path = (workspace / 'memory' / 'translation_engine.sqlite').resolve()
connection = sqlite3.connect(f"file:{db_path.as_posix()}?mode=ro", uri=True, timeout=90)
connection.row_factory = sqlite3.Row
print('DB:', db_path.relative_to(workspace))

DB: memory\translation_engine.sqlite


In [2]:
latest = dict(connection.execute("""
SELECT p.version_number AS version,
       p.changed_from_parent_count AS package_changes,
       ROUND(p.full_average_score, 9) AS package_score,
       e.id AS epoch, e.status AS epoch_status,
       e.baseline_tree_hash = e.output_tree_hash AS same_package_trees,
       c.decision AS calibration_decision,
       c.candidate_count AS calibration_candidates,
       d.actionable_family_count AS actionable_families,
       d.ignored_score_only_count AS ignored_score_only
FROM package_versions p
JOIN quality_epochs e ON e.id = 13
JOIN ml_pairwise_calibration_policy_decisions c ON c.id = 11
JOIN ml_quality_pattern_discovery_runs d ON d.id = 13
WHERE p.id = 6
""").fetchone())
latest['same_package_trees'] = bool(latest['same_package_trees'])
print(json.dumps(latest, ensure_ascii=False, indent=2))

{
  "version": 6,
  "package_changes": 118,
  "package_score": 0.771022041,
  "epoch": 13,
  "epoch_status": "scored",
  "same_package_trees": true,
  "calibration_decision": "skip",
  "calibration_candidates": 0,
  "actionable_families": 0,
  "ignored_score_only": 43112
}


## Results

In [3]:
pairwise = dict(connection.execute("""
SELECT COUNT(*) AS package_change_count,
       SUM(score_delta > 0) AS pairwise_improved,
       SUM(score_delta < 0) AS pairwise_regressed,
       ROUND(AVG(score_delta), 9) AS pairwise_mean_delta
FROM package_version_changes WHERE version_id = 6
""").fetchone())

rerun = dict(connection.execute("""
SELECT SUM(a.candidate_text = b.candidate_text) AS same_text_count_382_383,
       SUM(ABS(a.model_safe_probability-b.model_safe_probability) > 1e-12) AS score_changed_count_382_383,
       SUM((a.model_safe_probability < .5) <> (b.model_safe_probability < .5)) AS score_crossed_50_count,
       ROUND(AVG(ABS(a.model_safe_probability-b.model_safe_probability)), 9) AS post_materialization_mean_abs_delta,
       ROUND(MAX(b.model_safe_probability-a.model_safe_probability), 9) AS post_materialization_max_delta
FROM ml_score_items a
JOIN ml_score_items b ON b.run_id = 383 AND b.segment_id = a.segment_id
WHERE a.run_id = 382
""").fetchone())

scores = {row['run_id']: row['score'] for row in connection.execute("""
SELECT run_id, ROUND(AVG(model_safe_probability), 9) AS score
FROM ml_score_items WHERE run_id IN (382, 383) GROUP BY run_id
""")}
result = {**pairwise, **rerun, 'score_382': scores[382], 'score_383': scores[383]}
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "package_change_count": 118,
  "pairwise_improved": 118,
  "pairwise_regressed": 0,
  "pairwise_mean_delta": 0.023948585,
  "same_text_count_382_383": 287895,
  "score_changed_count_382_383": 118,
  "score_crossed_50_count": 52,
  "post_materialization_mean_abs_delta": 8.8275e-05,
  "post_materialization_max_delta": 0.407233,
  "score_382": 0.771022041,
  "score_383": 0.771110316
}


In [4]:
top_issues = {row['issue_code']: row['below_50'] for row in connection.execute("""
SELECT json_extract(j.value, '$.code') AS issue_code,
       SUM(i.model_safe_probability < .5) AS below_50
FROM ml_score_items i, json_each(i.issues_json) j
WHERE i.run_id = 383
GROUP BY issue_code
ORDER BY below_50 DESC
LIMIT 6
""")}
print(json.dumps(top_issues, ensure_ascii=False, indent=2))

{
  "spanish_residue": 739,
  "space_before_punctuation": 692,
  "spanish_residue_in_literal": 584,
  "gender_token_extra_prefix": 530,
  "missing_space_after_token": 441,
  "replacement_question_mark_mojibake": 353
}


In [5]:
integrity = {row['run_id']: (row['rows'], row['segments']) for row in connection.execute("""
SELECT run_id, COUNT(*) AS rows, COUNT(DISTINCT segment_id) AS segments
FROM ml_score_items WHERE run_id IN (381, 382, 383) GROUP BY run_id
""")}
assert all(rows == segments == 287895 for rows, segments in integrity.values())
assert result['package_change_count'] == result['pairwise_improved'] == 118
assert result['pairwise_regressed'] == 0
assert result['same_text_count_382_383'] == 287895
assert result['score_changed_count_382_383'] == 118
assert latest['same_package_trees'] and latest['calibration_candidates'] == 0
print('Checks passed: complete grain, positive v6 bridge, and baseline-sensitive rerun detected.')
connection.close()

Checks passed: complete grain, positive v6 bridge, and baseline-sensitive rerun detected.


## Takeaways

1. Preservar o comparativo pareado como prova da melhoria da v6; não usar o salto do run 383 como ganho de qualidade.
2. Criar um score intrínseco invariável à troca de baseline e manter o score atual como risco operacional de promoção.
3. Encerrar automaticamente epochs sem delta (`baseline_hash == output_hash`) para não deixar o gate de materialização aberto.
4. Separar `release_closed` de `quality_debt`: há 3.055 segmentos abaixo de 50% com issue explícita, mas o discovery atual marca todas as famílias recentes como `closed_observation`.
5. Priorizar providers de resíduos espanhóis e pontuação/espaçamento com calibração por família e rollout em shadow.